In [1]:
import torch
import torch.nn as nn
import torch.optim as optim 
import torchvision
from torchvision.datasets import MNIST
from torchvision.transforms import transforms

In [2]:
# Transform data
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5),(0.5))
])

In [3]:
# download datasets
trainset = MNIST(root = "./MNIST",train = True,download = True,transform = transform)
testset = MNIST(root = "./MNIST",train = False, download = True,transform = transform)

In [4]:
trainset

Dataset MNIST
    Number of datapoints: 60000
    Root location: ./MNIST
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=0.5, std=0.5)
           )

In [5]:
testset

Dataset MNIST
    Number of datapoints: 10000
    Root location: ./MNIST
    Split: Test
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=0.5, std=0.5)
           )

In [6]:
# the create of data loader

from torch.utils.data import DataLoader
train_loader= DataLoader(trainset,batch_size = 64,shuffle = True)
test_loader = DataLoader(testset,batch_size = 64)

## the build of RNN ERCHITECTURE

In [11]:
class RNN(nn.Module):
    def __init__(self):
        super(RNN,self).__init__()
        self.rnn = nn.RNN(
            input_size = 28,
            hidden_size = 128,
            num_layers = 2,
            batch_first = True
        )
        # fully connected layer
        self.fc =  nn.Linear(128,10)
    def forward(self,x):

        output, hidden = self.rnn(x)
        output = output[:,-1,:]
        output = self.fc(output)

        return output

In [12]:
model = RNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

## Training

In [15]:
epochs = 10

train_losses = []
train_accuracies = []

for epoch in range(epochs):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        # Convert image to sequence
        images = images.squeeze(1)

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)

        # Calculate loss
        loss = criterion(outputs, labels)

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        # Add loss
        running_loss += loss.item()

        # Predictions
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_loader)

    epoch_accuracy = 100 * correct / total

    train_losses.append(epoch_loss)

    train_accuracies.append(epoch_accuracy)

    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"Loss: {epoch_loss:.4f} "
        f"Accuracy: {epoch_accuracy:.2f}%"
    )

Epoch [1/10] Loss: 0.2833 Accuracy: 91.75%
Epoch [2/10] Loss: 0.2168 Accuracy: 93.80%
Epoch [3/10] Loss: 0.1799 Accuracy: 94.82%
Epoch [4/10] Loss: 0.1673 Accuracy: 95.21%
Epoch [5/10] Loss: 0.1529 Accuracy: 95.74%
Epoch [6/10] Loss: 0.1354 Accuracy: 96.18%
Epoch [7/10] Loss: 0.1199 Accuracy: 96.60%
Epoch [8/10] Loss: 0.1295 Accuracy: 96.30%
Epoch [9/10] Loss: 0.1294 Accuracy: 96.29%
Epoch [10/10] Loss: 0.1165 Accuracy: 96.72%


In [16]:
# Evalution process 

model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0
    
    for Xb, yb in test_loader:
        Xb = Xb.squeeze(1)

        outputs = model(Xb)
        predicted = torch.max(outputs,1)

        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f"accuracy = {correct_vals/tot_vals*100}")

AttributeError: 'bool' object has no attribute 'sum'